# CNN Image Classification (MNIST) - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، مع شرح تفصيلي لكل خطوة: ماذا نفعل، ولماذا، وكيف نفسر النتائج للطلاب.

## الهدف التعليمي
تصنيف أرقام مكتوبة بخط اليد (MNIST sample) باستخدام **Convolutional Neural Network**.

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات
3. تجهيز X و y (reshape + normalize)
4. تقسيم البيانات (stratify)
5. بناء CNN
6. compile + fit
7. التقييم
8. عرض تنبؤات


## الخطوة 1: استيراد المكتبات

Conv2D يستخرج features مكانية؛ MaxPooling2D يصغّر الأبعاد.


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:

- **`numpy` (المستوردة كـ `np`):** للعمليات الحسابية والتعامل مع المصفوفات الرياضية.
- **`pandas` (المستوردة كـ `pd`):** لقراءة البيانات وإدارة الجداول البرمجية (DataFrames).
- **`matplotlib.pyplot` (المستوردة كـ `plt`):** للرسم البياني وتصور البيانات بصرياً.
- **`train_test_split`:** لتقسيم البيانات إلى مجموعة تدريب ومجموعة اختبار بشكل عشوائي ومنظم.
- **`tensorflow / keras`:** لبناء وتدريب الشبكات العصبية الاصطناعية ونماذج التعلم العميق.


In [ ]:
# الخطوة 1) استيراد المكتبات
# pip install tensorflow -q
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense


## الخطوة 2: قراءة البيانات

500 صورة × 784 بكسل + عمود label.
كل صورة 28×28 بكسل (قيم 0–255).


### أولاً: استيراد المكتبات البرمجية المطلوبة (Import Libraries)
نقوم في هذه الخطوة باستيراد الأدوات والمكتبات اللازمة لمعالجة البيانات، بناء وتدريب النموذج، وتقييم النتائج:




In [ ]:
# Step 2) قراءة البيانات / Load dataset
import os
import urllib.request

filename = 'mnist_sample.csv'
if not os.path.exists(filename):
    url = 'https://raw.githubusercontent.com/iksasa15/AI-ML/main/code/15-%20Deep%20Learning/4-%20CNN%20Image%20Classification/mnist_sample.csv'
    urllib.request.urlretrieve(url, filename)

dataset = pd.read_csv(filename)
dataset.head()


## الخطوة 3: تجهيز الصور

- reshape إلى (n, 28, 28, 1) — batch, height, width, channels
- `/255.0` — normalization إلى [0, 1] لتسريع التدريب


### خطوة: الخطوة 3) تجهيز X و y
نقوم بتشغيل هذا الجزء من الكود لتنفيذ العمليات البرمجية الموضحة في التعليقات أعلاه لتجهيز البيانات أو تهيئة النموذج.


In [ ]:
# الخطوة 3) تجهيز X و y
y = dataset['label'].values.astype(int)
pixel_cols = [c for c in dataset.columns if c.startswith('pixel_')]
X = dataset[pixel_cols].values.reshape(-1, 28, 28, 1).astype('float32') / 255.0
print('X shape:', X.shape)
print('y shape:', y.shape)


## الخطوة 4: تقسيم البيانات

`stratify=y` يحافظ على نسب الفئات (0–9) في التدريب والاختبار.


### ثامناً: تقسيم البيانات إلى مجموعتي تدريب واختبار (Train/Test Split)
نقسم البيانات بنسبة 20% لمجموعة الاختبار وبقية البيانات لمجموعة التدريب:
- **بيانات التدريب (Training Set):** لتعليم النموذج وضبط أوزانه ومعاملاته.
- **بيانات الاختبار (Test Set):** لتقييم النموذج واختبار قدرته على التنبؤ ببيانات جديدة كلياً.


In [ ]:
# الخطوة 4) تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)


## الخطوة 5: بناء CNN

1. **Conv2D(32, 3×3)**: 32 filter يمسح الصورة
2. **MaxPooling2D(2×2)**: يأخذ max من كل 2×2 → يصغّر
3. **Flatten**: يحوّل tensor لمتجه
4. **Dense(128) + Dense(10, softmax)**: 10 فئات (أرقام 0–9)


### عاشراً: بناء وتدريب نموذج الشبكة العصبية الاصطناعية (Neural Network)
1. نقوم بإنشاء كائن من النموذج بالمعاملات المناسبة.
2. نستخدم الدالة `.fit(X_train, y_train)` لتدريب النموذج على بيانات التدريب لكي يتعلم العلاقات والأنماط.


In [ ]:
# الخطوة 5) بناء CNN
model = Sequential([
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(units=128, activation='relu'),
    Dense(units=10, activation='softmax')
])
model.summary()


## الخطوة 6: compile + fit

- **sparse_categorical_crossentropy**: y أعداد صحيحة (0–9) وليست one-hot
- **softmax**: احتمالات 10 فئات ت sum = 1


### خطوة: الخطوة 6) compile + fit
نقوم بتشغيل هذا الجزء من الكود لتنفيذ العمليات البرمجية الموضحة في التعليقات أعلاه لتجهيز البيانات أو تهيئة النموذج.


In [ ]:
# الخطوة 6) compile + fit
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train, y_train,
    batch_size=32,
    epochs=15,
    validation_split=0.2,
    verbose=1
)


## الخطوة 7: التقييم

نقيّم على test set المنفصل.


### خطوة: الخطوة 7) التقييم
نقوم بتشغيل هذا الجزء من الكود لتنفيذ العمليات البرمجية الموضحة في التعليقات أعلاه لتجهيز البيانات أو تهيئة النموذج.


In [ ]:
# الخطوة 7) التقييم
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test loss: {loss:.4f}')
print(f'Test accuracy: {accuracy:.2%}')


## الخطوة 8: عرض التنبؤات

أخضر = تنبؤ صحيح، أحمر = خطأ.
`argmax` يختار الفئة ذات أعلى احتمال.


### الحادي عشر: التنبؤ بقيم مجموعة الاختبار (Make Predictions)
نستخدم النموذج المدرب للتنبؤ بالنتائج للمدخلات الموجودة في مجموعة الاختبار للتأكد من قدرة النموذج على التعميم على بيانات جديدة لم يتدرب عليها من قبل.


In [ ]:
# الخطوة 8) عرض صور + تنبؤات
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    color = 'green' if y_pred[i] == y_test[i] else 'red'
    ax.set_title(f'True: {y_test[i]} | Pred: {y_pred[i]}', color=color)
    ax.axis('off')
plt.suptitle('Sample Predictions (green=correct, red=wrong)')
plt.tight_layout()
plt.show()
